# U2T01 — Named Entity Recognition (CoNLL-2003)

# Task2: NER · **Base model:** `bert-base-cased`




In [3]:
%pip install "transformers==4.49.0" "datasets==3.5.1" "accelerate==1.5.2" "huggingface_hub>=0.29,<1" "nbformat>=5"
%pip install --use-pep517 "seqeval==1.2.2"

  Using cached transformers-4.49.0-py3-none-any.whl.metadata (44 kB)
  Using cached datasets-3.5.1-py3-none-any.whl.metadata (19 kB)
  Using cached accelerate-1.5.2-py3-none-any.whl.metadata (19 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 121.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025

In [4]:
import os, sys, json, time, platform, importlib.metadata as md
from pathlib import Path
import numpy as np
import torch
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/UPY_TDS_NER_Diego')
ROOT.mkdir(parents=True, exist_ok=True)
SEED=42; BASE='bert-base-cased'; DATASET='lhoestq/conll2003'; MAX_LEN=256
EPOCHS=3; TRAIN_BS=16; EVAL_BS=32
assert torch.cuda.is_available(), 'Enable Runtime > Change runtime type > GPU, then restart and rerun.'
print('GPU:', torch.cuda.get_device_name(0), 'ROOT:', ROOT)
versions={p:md.version(p) for p in ['torch','transformers','datasets','accelerate','seqeval','huggingface_hub']}
(ROOT/'environment.json').write_text(json.dumps(dict(versions=versions,python=sys.version,gpu=torch.cuda.get_device_name(0),seed=SEED,base=BASE,dataset=DATASET,max_length=MAX_LEN,epochs=EPOCHS,batch_size=TRAIN_BS),indent=2))
print(versions)

Mounted at /content/drive
GPU: NVIDIA L4 ROOT: /content/drive/MyDrive/UPY_TDS_NER_Diego
{'torch': '2.11.0+cu128', 'transformers': '4.49.0', 'datasets': '3.5.1', 'accelerate': '1.5.2', 'seqeval': '1.2.2', 'huggingface_hub': '0.36.2'}


## 1. Load and inspect the official splits
Never tune on test. Keep the full training set (14,041 examples), validation (3,250) and test (3,453), subject to what the downloaded revision actually reports.

In [7]:
from datasets import load_dataset
from transformers import set_seed

set_seed(SEED)
raw = load_dataset(DATASET)

print(raw)
print(raw["train"].features)

label_names = [
    "O", "B-PER", "I-PER", "B-ORG", "I-ORG",
    "B-LOC", "I-LOC", "B-MISC", "I-MISC"
]
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in id2label.items()}

assert all(
    0 <= tag < len(label_names)
    for split in raw
    for row in raw[split]["ner_tags"]
    for tag in row
)
assert raw["train"][0]["tokens"][0] == "EU"
assert id2label[raw["train"][0]["ner_tags"][0]] == "B-ORG"

print("Labels:", label_names)
print(
    "Example:",
    list(zip(
        raw["train"][0]["tokens"],
        [id2label[i] for i in raw["train"][0]["ner_tags"]]
    ))
)

(ROOT / "dataset_metadata.json").write_text(
    json.dumps({
        "dataset": DATASET,
        "sizes": {k: len(v) for k, v in raw.items()},
        "labels": label_names,
        "features": str(raw["train"].features)
    }, indent=2)
)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})
{'id': Value(dtype='string', id=None), 'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'pos_tags': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None), 'chunk_tags': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None), 'ner_tags': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None)}
Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
Example: [('EU', 'B-ORG'), ('rejects', 'O'), ('German', 'B-MISC'), ('call', 'O'), ('to', 'O'), ('boycott', 'O'), ('British', 'B-MISC'), ('lamb', 'O'), 

629

## 2. Align word labels to BERT subtokens
`-100` removes specials, continuation subtokens and padding from loss **and metrics**. Sanity-check a real padded batch.

In [8]:
from transformers import DataCollatorForTokenClassification
tokenizer=AutoTokenizer.from_pretrained(BASE, use_fast=True)
assert tokenizer.is_fast

def align(batch):
    enc=tokenizer(batch['tokens'],is_split_into_words=True,truncation=True,max_length=MAX_LEN)
    aligned=[]
    for i,tags in enumerate(batch['ner_tags']):
        previous=None; row=[]
        for word_id in enc.word_ids(batch_index=i):
            row.append(tags[word_id] if word_id is not None and word_id != previous else -100)
            previous=word_id
        assert len(row)==len(enc['input_ids'][i])
        aligned.append(row)
    enc['labels']=aligned
    return enc

tok=raw.map(align,batched=True,remove_columns=raw['train'].column_names)
collator=DataCollatorForTokenClassification(tokenizer)
batch=collator([tok['train'][0],tok['train'][4]])
assert batch['input_ids'].shape == batch['labels'].shape
print('token / label from first padded batch:')
for t,l in zip(tokenizer.convert_ids_to_tokens(batch['input_ids'][0]),batch['labels'][0].tolist()):
    print(f'{t:20s} {id2label[l] if l!=-100 else "-100"}')
assert (batch['labels'][0][batch['attention_mask'][0]==0]==-100).all()
# Count words omitted due to truncation, for limitations.
truncated=0
for split in raw:
    counts=[len(x['ner_tags'])-sum(y!=-100 for y in tok[split][j]['labels']) for j,x in enumerate(raw[split])]
    truncated+=sum(n>0 for n in counts)
    print(split, 'sentences truncated:',sum(n>0 for n in counts),'omitted words:',sum(counts))
print('All splits, sentences truncated:',truncated)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

token / label from first padded batch:
[CLS]                -100
EU                   B-ORG
rejects              O
German               B-MISC
call                 O
to                   O
boycott              O
British              B-MISC
la                   O
##mb                 -100
.                    O
[SEP]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]                -100
[PAD]       

## 3. Metric: strict IOB2 entity-level precision, recall, F1
F1 scores complete spans and entity types; token accuracy can look high because `O` is frequent. Validation F1 selects the winner. Accuracy is a secondary diagnostic.

In [9]:
from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report
from seqeval.scheme import IOB2

def decode(logits, labels):
    ids=np.argmax(logits,axis=-1)
    pred=[[id2label[int(p)] for p,l in zip(pr,lab) if l!=-100] for pr,lab in zip(ids,labels)]
    gold=[[id2label[int(l)] for l in lab if l!=-100] for lab in labels]
    return pred,gold

def compute_metrics(eval_pred):
    pred,gold=decode(eval_pred.predictions,eval_pred.label_ids)
    return {'precision':precision_score(gold,pred,mode='strict',scheme=IOB2),
            'recall':recall_score(gold,pred,mode='strict',scheme=IOB2),
            'f1':f1_score(gold,pred,mode='strict',scheme=IOB2),
            'token_accuracy':accuracy_score(gold,pred)}

## 4. Run both methods
Same pretrained checkpoint, head initialization seed, train/validation data, max length, batch size and epochs. The frozen head gets `1e-3`; when the encoder trains, it gets `2e-5` while the head keeps `1e-3`. An epoch history and total runtime are saved for the report.

In [10]:
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments

class DifferentialLRTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is None:
            no_decay=('bias','LayerNorm.weight')
            groups=[]
            for prefix,lr in [('bert',2e-5),('classifier',1e-3)]:
                for decay in [True,False]:
                    params=[p for name,p in self.model.named_parameters() if p.requires_grad and name.startswith(prefix+'.') and (not any(x in name for x in no_decay))==decay]
                    if params:
                        groups.append({'params':params,'lr':lr,'weight_decay':0.01 if decay else 0.0})
            assert sum(p.numel() for g in groups for p in g['params']) == sum(p.numel() for p in self.model.parameters() if p.requires_grad)
            self.optimizer=torch.optim.AdamW(groups)
        return self.optimizer

def run(method):
    assert method in ('frozen','full')
    set_seed(SEED)
    directory=ROOT/method
    directory.mkdir(exist_ok=True)
    model=AutoModelForTokenClassification.from_pretrained(BASE,num_labels=len(label_names),id2label=id2label,label2id=label2id)
    if method=='frozen':
        for p in model.bert.parameters(): p.requires_grad=False
    counts=dict(total=sum(p.numel() for p in model.parameters()),trainable=sum(p.numel() for p in model.parameters() if p.requires_grad),encoder=sum(p.numel() for p in model.bert.parameters() if p.requires_grad))
    print(method, counts)
    assert (counts['encoder']==0) == (method=='frozen')
    args=TrainingArguments(output_dir=str(directory/'checkpoints'),num_train_epochs=EPOCHS,per_device_train_batch_size=TRAIN_BS,per_device_eval_batch_size=EVAL_BS,
        eval_strategy='epoch',save_strategy='epoch',logging_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='f1',greater_is_better=True,
        save_total_limit=2,seed=SEED,data_seed=SEED,fp16=True,report_to='none',lr_scheduler_type='constant',
        push_to_hub=False)
    trainer=DifferentialLRTrainer(model=model,args=args,train_dataset=tok['train'],eval_dataset=tok['validation'],data_collator=collator,processing_class=tokenizer,compute_metrics=compute_metrics)
    t=time.perf_counter()
    train_output=trainer.train()
    seconds=time.perf_counter()-t
    val=trainer.evaluate(tok['validation'])
    history=trainer.state.log_history
    record={'method':method,'counts':counts,'train_runtime_wall_seconds':seconds,'train_metrics':train_output.metrics,'validation':val,
        'best_checkpoint':trainer.state.best_model_checkpoint,'best_validation_f1':trainer.state.best_metric,'log_history':history,
        'config':{'seed':SEED,'base':BASE,'dataset':DATASET,'epochs':EPOCHS,'batch':TRAIN_BS,'max_len':MAX_LEN,'head_lr':1e-3,'encoder_lr':2e-5 if method=='full' else None}}
    (directory/'results.json').write_text(json.dumps(record,indent=2,default=float))
    trainer.save_model(str(directory/'best_model'))
    tokenizer.save_pretrained(str(directory/'best_model'))
    print(method,'validation F1:',val['eval_f1'],'seconds:',round(seconds,1))
    del trainer,model
    torch.cuda.empty_cache()
    return record

frozen=run('frozen')

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


frozen {'total': 107726601, 'trainable': 6921, 'encoder': 0}


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Token Accuracy
1,0.240000,0.134040,0.852695,0.806631,0.829024,0.967933
2,0.127800,0.112423,0.867835,0.833221,0.850176,0.972918
3,0.114600,0.106833,0.872288,0.839111,0.855378,0.973833


frozen validation F1: 0.8553782810087494 seconds: 69.6


In [11]:
full=run('full')

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


full {'total': 107726601, 'trainable': 107726601, 'encoder': 107719680}


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Token Accuracy
1,0.102000,0.037642,0.943294,0.935039,0.939148,0.989272
2,0.025600,0.039463,0.945473,0.945473,0.945473,0.990518
3,0.012200,0.044045,0.943270,0.945809,0.944538,0.990538


full validation F1: 0.9454729047458769 seconds: 254.7


## 5. Compare validation results and test the chosen model once
If the F1 gap is small (~0.01–0.03), call it inconclusive after one seed. Consider model size, train time and course team coverage when selecting the deliverable. The cell below chooses higher validation F1; you may change `winner` with a written justification before running the test.

In [12]:
import pandas as pd
records=[frozen,full]
summary=pd.DataFrame([{'method':r['method'],'trainable_parameters':r['counts']['trainable'],'validation_precision':r['validation']['eval_precision'],'validation_recall':r['validation']['eval_recall'],'validation_f1':r['validation']['eval_f1'],'validation_token_accuracy':r['validation']['eval_token_accuracy'],'train_seconds':r['train_runtime_wall_seconds'],'train_steps':r['train_metrics']['train_steps_per_second']*r['train_metrics']['train_runtime'],'seconds_per_step_approx':1/r['train_metrics']['train_steps_per_second']} for r in records])
summary.to_csv(ROOT/'validation_comparison.csv',index=False)
display(summary)
winner=max(records,key=lambda r:r['validation']['eval_f1'])['method']
print('Validation winner:',winner,'; gap:',abs(frozen['validation']['eval_f1']-full['validation']['eval_f1']))

,method,trainable_parameters,validation_precision,validation_recall,validation_f1,validation_token_accuracy,train_seconds,train_steps,seconds_per_step_approx
0,frozen,6921,0.872288,0.839111,0.855378,0.973833,69.633581,2634.035335,0.026276
1,full,107726601,0.945473,0.945473,0.945473,0.990518,254.690823,2634.079269,0.096534


Validation winner: full ; gap: 0.09009462373712751


In [13]:
from transformers import AutoModelForTokenClassification
chosen=ROOT/winner/'best_model'
model=AutoModelForTokenClassification.from_pretrained(str(chosen))
args=TrainingArguments(output_dir=str(ROOT/'test_only'),per_device_eval_batch_size=EVAL_BS,report_to='none',fp16=True)
tester=Trainer(model=model,args=args,data_collator=collator,processing_class=tokenizer,compute_metrics=compute_metrics)
test_output=tester.predict(tok['test'])
pred,gold=decode(test_output.predictions,test_output.label_ids)
test_result=dict(test_output.metrics)
test_result['classification_report']=classification_report(gold,pred,mode='strict',scheme=IOB2,output_dict=True)
test_result['winner']=winner
(ROOT/'final_test.json').write_text(json.dumps(test_result,indent=2,default=float))
print('TEST: ',{k:v for k,v in test_result.items() if k!='classification_report'})
print(classification_report(gold,pred,mode='strict',scheme=IOB2))
# Qualitative example: predictions on a held-out sentence, kept for error analysis.
idx=0
print(list(zip(raw['test'][idx]['tokens'],gold[idx],pred[idx])))
print('Saved:',ROOT)

TEST:  {'test_loss': 0.1058770939707756, 'test_model_preparation_time': 0.003, 'test_precision': 0.9112603853632667, 'test_recall': 0.9127124645892352, 'test_f1': 0.911985846970367, 'test_token_accuracy': 0.9824485840422096, 'test_runtime': 2.6733, 'test_samples_per_second': 1291.655, 'test_steps_per_second': 40.399, 'winner': 'full'}
              precision    recall  f1-score   support

         LOC       0.93      0.93      0.93      1668
        MISC       0.84      0.81      0.82       702
         ORG       0.88      0.90      0.89      1661
         PER       0.95      0.95      0.95      1617

   micro avg       0.91      0.91      0.91      5648
   macro avg       0.90      0.90      0.90      5648
weighted avg       0.91      0.91      0.91      5648

[('SOCCER', 'O', 'O'), ('-', 'O', 'O'), ('JAPAN', 'B-LOC', 'B-PER'), ('GET', 'O', 'O'), ('LUCKY', 'O', 'B-PER'), ('WIN', 'O', 'O'), (',', 'O', 'O'), ('CHINA', 'B-PER', 'B-PER'), ('IN', 'O', 'O'), ('SURPRISE', 'O', 'O'), ('DEFEAT

## 6. Publish the selected model (run only after reviewing metrics and checking data terms)
Create a Hugging Face **write** token in Settings > Access Tokens. Do not paste it into notebook source; Colab will prompt securely. This publishes the BERT weights, tokenizer and model card. Choose public or private. If private, grant `Dexterg83` access on the model repository from its Settings page. Keep the notebook and result JSON/CSV in your team deliverables. You can defer this cell until the team agrees on the final model.

In [14]:
# Run this cell only when ready to publish.
from huggingface_hub import login, whoami, HfApi
from getpass import getpass
login(token=getpass('HF write token (input hidden): '),add_to_git_credential=False)
repo_id=f"{whoami()['name']}/upy-tds-bert-ner-conll2003"
private=True # set False if your team decides on a public model
card=f"""---
language: en
pipeline_tag: token-classification
tags:
- bert
- named-entity-recognition
- conll2003
metrics:
- f1
model-index:
- name: {repo_id.split('/')[-1]}
  results: []
---
# BERT NER on CoNLL-2003

## Intended use
English newswire named entity recognition (PER, ORG, LOC, MISC) for a course experiment. Not a general-domain or safety-critical extractor.

## Model and training
- Base: [{BASE}](https://huggingface.co/{BASE})
- Dataset: [{DATASET}](https://huggingface.co/datasets/{DATASET}); splits train/validation/test; original shared task: [CoNLL-2003](https://aclanthology.org/W03-0419/).
- Adaptation: {winner}; initial seed {SEED}; {EPOCHS} epochs; batch {TRAIN_BS}; maximum sequence length {MAX_LEN}.
- Two compared alternatives: frozen BERT with trained token head and full fine-tuning. Head learning rate 1e-3; encoder learning rate 2e-5 when trained.
- Labels: {', '.join(label_names)}. Supervise the first subtoken only; other positions -100.

## Results
Entity-level strict IOB2 span F1 on held-out test: {test_result['test_f1']:.4f}; precision {test_result['test_precision']:.4f}; recall {test_result['test_recall']:.4f}; token accuracy {test_result['test_token_accuracy']:.4f}. Validation F1: {max(records,key=lambda r:r['validation']['eval_f1'])['validation']['eval_f1']:.4f}. Single seed; close differences may be noise.

## Limitations
English Reuters news from 1996; domain and time shift can hurt performance. One label per word; truncated sentences beyond {MAX_LEN} subtokens lose supervised words. Named entities and noisy labels may differ across datasets. Review the upstream dataset terms before reuse or redistribution.

## References
- Tjong Kim Sang & De Meulder (2003), [CoNLL-2003](https://aclanthology.org/W03-0419/).
- Devlin et al. (2019), [BERT](https://arxiv.org/abs/1810.04805).
- [Hugging Face token classification](https://huggingface.co/docs/transformers/tasks/token_classification).
"""
(chosen/'README.md').write_text(card)
api=HfApi()
api.create_repo(repo_id=repo_id,private=private,exist_ok=True)
api.upload_folder(repo_id=repo_id,folder_path=str(chosen),repo_type='model')
print('Published:',f'https://huggingface.co/{repo_id}')

HF write token (input hidden): ··········


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t_model/model.safetensors:   1%|          | 3.53MB /  431MB            

  ...t_model/training_args.bin:   2%|2         |   142B / 5.78kB            

Published: https://huggingface.co/Perry-DLC/upy-tds-bert-ner-conll2003
